In [1]:
# ============================================================
# 12) Evaluate MedGemma pseudo-reports directly against ground truth
#
# Logic:
# - Read pseudo-reports from:
#     /data/liangz2/openi/biomedclip_mimic_13label_cache/pseudo_reports.jsonl
# - For each record, get image_path
# - Derive paired GT json path by replacing ".jpg" -> ".json"
# - GT labels come from GT json["labels"]
#   * if empty => all 13 labels = 0, and normal = "yes"
# - Predicted labels come from pseudo_report text after "PREDICTED LABELS:"
#   * if "Normal chest X-ray." => all 13 labels = 0, normal = "yes"
#   * otherwise parse comma-separated labels and convert to multi-hot
# - Compute:
#     Macro F1, Micro F1, Hamming Accuracy, Exact Match,
#     Macro Sensitivity, Macro Specificity, Macro Youden-J,
#     ROC-AUC (not applicable here -> NaN unless score proxy used),
#     FER, FER for abnormal studies, Omission Rate
# ============================================================

from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional
import json
import re
import numpy as np
from sklearn.metrics import (
    f1_score,
    hamming_loss,
    multilabel_confusion_matrix,
    roc_auc_score,
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
PSEUDO_REPORT_JSONL = Path("/data/liangz2/openi/biomedclip_mimic_13label_cache/pseudo_reports.jsonl")

# ------------------------------------------------------------
# 13-label space (must match notebook section 0)
# ------------------------------------------------------------
LABELS_13 = [
    "atelectasis",
    "cardiomegaly",
    "consolidation",
    "edema",
    "enlarged cardiomediastinum",
    "fracture",
    "lung lesion",
    "lung opacity",
    "pleural effusion",
    "pleural other",
    "pneumonia",
    "pneumothorax",
    "support devices",
]

LABEL_TO_INDEX = {lab: i for i, lab in enumerate(LABELS_13)}

# ------------------------------------------------------------
# Label normalization / alias mapping
# ------------------------------------------------------------
def _normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    s = s.strip(" .;:,\n\t")
    return s

LABEL_ALIASES = {
    # canonical -> canonical
    "atelectasis": "atelectasis",
    "cardiomegaly": "cardiomegaly",
    "consolidation": "consolidation",
    "edema": "edema",
    "enlarged cardiomediastinum": "enlarged cardiomediastinum",
    "fracture": "fracture",
    "lung lesion": "lung lesion",
    "lung opacity": "lung opacity",
    "pleural effusion": "pleural effusion",
    "pleural other": "pleural other",
    "pneumonia": "pneumonia",
    "pneumothorax": "pneumothorax",
    "support devices": "support devices",

    # common variants / synonyms that may appear in pseudo reports
    "pulmonary edema": "edema",
    "cardiomediastinal enlargement": "enlarged cardiomediastinum",
    "mediastinal enlargement": "enlarged cardiomediastinum",
    "cardiomediastinal widening": "enlarged cardiomediastinum",
    "widened mediastinum": "enlarged cardiomediastinum",
    "pleural effusions": "pleural effusion",
    "support device": "support devices",
    "medical device": "support devices",
    "medical devices": "support devices",
    "tubes and lines": "support devices",
    "line/tube": "support devices",
    "lines/tubes": "support devices",
    "opacity": "lung opacity",
    "lung opacities": "lung opacity",
    "pulmonary opacity": "lung opacity",
    "pulmonary opacities": "lung opacity",
    "lesion": "lung lesion",
    "pulmonary lesion": "lung lesion",
    "rib fracture": "fracture",
    "osseous fracture": "fracture",
}

def canonicalize_label(label: str) -> Optional[str]:
    lab = _normalize_text(label)
    if lab in LABEL_ALIASES:
        return LABEL_ALIASES[lab]

    # lightweight fuzzy containment for known patterns
    if "pulmonary edema" in lab:
        return "edema"
    if "cardiomegaly" in lab:
        return "cardiomegaly"
    if "atelectasis" in lab:
        return "atelectasis"
    if "consolidation" in lab:
        return "consolidation"
    if "pneumonia" in lab:
        return "pneumonia"
    if "pneumothorax" in lab:
        return "pneumothorax"
    if "pleural effusion" in lab:
        return "pleural effusion"
    if "pleural other" in lab:
        return "pleural other"
    if "lung lesion" in lab or ("lesion" in lab and "lung" in lab):
        return "lung lesion"
    if "lung opacity" in lab or "pulmonary opacity" in lab or "opacity" == lab:
        return "lung opacity"
    if "support device" in lab or "medical device" in lab or "tube" in lab or "line" in lab:
        return "support devices"
    if "fracture" in lab:
        return "fracture"
    if "enlarged cardiomediastinum" in lab or "cardiomediastinum" in lab or "mediastinum" in lab:
        return "enlarged cardiomediastinum"

    return None

In [2]:
# ------------------------------------------------------------
# IO helpers
# ------------------------------------------------------------
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def image_path_to_json_path(image_path: str) -> Path:
    p = Path(image_path)
    if p.suffix.lower() != ".json":
        p = p.with_suffix(".json")
    return p

In [3]:
# ------------------------------------------------------------
# GT label extraction from paired json
# ------------------------------------------------------------
def extract_gt_labels_binary_from_json_path(json_path: Path) -> Tuple[np.ndarray, str]:
    """
    GT labels are stored in json["labels"].
    If labels is empty, all labels = 0 and normal = "yes".
    Returns:
        y_true_vec: np.ndarray shape (13,)
        normal_str: "yes" or "no"
    """
    with open(json_path, "r", encoding="utf-8") as f:
        row = json.load(f)

    lab_raw = row.get("labels", [])
    if isinstance(lab_raw, str):
        try:
            lab_raw = json.loads(lab_raw)
        except Exception:
            lab_raw = [x.strip() for x in lab_raw.split(",") if x.strip()]

    if lab_raw is None:
        lab_raw = []

    lab_set = set()
    for lab in lab_raw:
        can = canonicalize_label(str(lab))
        if can is not None:
            lab_set.add(can)

    y = np.array([1 if lab in lab_set else 0 for lab in LABELS_13], dtype=np.int32)

    # follow user's requested logic
    normal_str = "yes" if len(lab_set) == 0 else str(row.get("normal", "no")).strip().lower()
    if len(lab_set) == 0:
        normal_str = "yes"
    elif normal_str not in {"yes", "no"}:
        normal_str = "no"

    return y, normal_str

In [4]:
# ------------------------------------------------------------
# Parse pseudo report -> predicted labels
# ------------------------------------------------------------
def extract_predicted_labels_text(pseudo_report: str) -> str:
    """
    Extract the text after 'PREDICTED LABELS'.
    Handles forms like:
      PREDICTED LABELS: Pneumonia, Pulmonary edema, Cardiomegaly
      PREDICTED LABELS - Normal chest X-ray.
    """
    if pseudo_report is None:
        return ""

    text = str(pseudo_report)

    # robust capture until end of string
    m = re.search(r"PREDICTED\s+LABELS\s*[:\-]\s*(.+)$", text, flags=re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).strip()

    # fallback: last line containing "predicted labels"
    for line in reversed(text.splitlines()):
        if "predicted labels" in line.lower():
            return re.sub(r"^.*predicted\s+labels\s*[:\-]\s*", "", line, flags=re.IGNORECASE).strip()

    return ""

def predicted_text_to_multihot(pred_text: str) -> Tuple[np.ndarray, str, List[str]]:
    """
    Convert predicted label text into:
      - 13-d multi-hot vector
      - normal_str ('yes'/'no')
      - canonical predicted labels list
    Rule:
      - If text contains 'Normal chest X-ray', all labels = 0 and normal = 'yes'
      - Else parse comma-separated predicted labels
    """
    pred_text = "" if pred_text is None else str(pred_text).strip()

    # exact / robust normal case
    if re.search(r"normal\s+chest\s*x[\-\s]?ray", pred_text, flags=re.IGNORECASE):
        return np.zeros(len(LABELS_13), dtype=np.int32), "yes", []

    # remove trailing commentary after first likely sentence break only if needed
    # but keep comma-separated items intact
    pred_text = pred_text.strip(" \n\t.;")

    # split by commas / semicolons / line breaks
    raw_items = re.split(r"[,;\n]+", pred_text)
    pred_labels: List[str] = []
    seen = set()

    for item in raw_items:
        item = _normalize_text(item)
        if not item:
            continue
        can = canonicalize_label(item)
        if can is not None and can not in seen:
            pred_labels.append(can)
            seen.add(can)

    y = np.array([1 if lab in seen else 0 for lab in LABELS_13], dtype=np.int32)
    normal_str = "yes" if y.sum() == 0 else "no"
    return y, normal_str, pred_labels

In [8]:
# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
def compute_comprehensive_metrics_hard_labels_only(
    y_true: np.ndarray,   # (N, 13) int
    y_pred: np.ndarray,   # (N, 13) int
    threshold: float = 0.5,
) -> Dict[str, Any]:
    N = y_true.shape[0]

    macro_f1    = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    micro_f1    = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    per_f1      = f1_score(y_true, y_pred, average=None, zero_division=0)
    hamming_acc = float(1.0 - hamming_loss(y_true, y_pred))
    exact_match = float((y_true == y_pred).all(axis=1).mean())

    total_fp_labels = int(((y_pred == 1) & (y_true == 0)).sum())
    total_fn_labels = int(((y_pred == 0) & (y_true == 1)).sum())
    total_predicted_positive_labels = int((y_pred == 1).sum())
    total_gt_positive_labels = int((y_true == 1).sum())

    fabrication_error_rate = (
        float(total_fp_labels / total_predicted_positive_labels)
        if total_predicted_positive_labels > 0 else 0.0
    )
    omission_rate = (
        float(total_fn_labels / total_gt_positive_labels)
        if total_gt_positive_labels > 0 else 0.0
    )

    abnormal_mask = (y_true.sum(axis=1) > 0)
    num_abnormal_cases = int(abnormal_mask.sum())

    abnormal_pred = y_pred[abnormal_mask]
    abnormal_true = y_true[abnormal_mask]

    abnormal_cases_with_pred_positive = (
        int((abnormal_pred.sum(axis=1) > 0).sum()) if num_abnormal_cases > 0 else 0
    )

    abnormal_fp_labels = (
        int(((abnormal_pred == 1) & (abnormal_true == 0)).sum()) if num_abnormal_cases > 0 else 0
    )
    abnormal_predicted_positive_labels = (
        int((abnormal_pred == 1).sum()) if num_abnormal_cases > 0 else 0
    )

    fer_abnormal = (
        float(abnormal_fp_labels / abnormal_predicted_positive_labels)
        if abnormal_predicted_positive_labels > 0 else 0.0
    )

    mcm = multilabel_confusion_matrix(y_true, y_pred)
    per_label: Dict[str, Any] = {}
    sens_l, spec_l, yj_l = [], [], []

    for i, lab in enumerate(LABELS_13):
        tn, fp, fn, tp = mcm[i].ravel()
        sens   = tp / (tp + fn + 1e-9) if (tp + fn) > 0 else float("nan")
        spec   = tn / (tn + fp + 1e-9) if (tn + fp) > 0 else float("nan")
        youden = (sens + spec - 1.0) if not (np.isnan(sens) or np.isnan(spec)) else float("nan")

        per_label[lab] = {
            "TP": int(tp),
            "FP": int(fp),
            "TN": int(tn),
            "FN": int(fn),
            "f1": float(per_f1[i]),
            "sensitivity": float(sens),
            "specificity": float(spec),
            "youden_j": float(youden),
            "roc_auc": float("nan"),   # intentionally unavailable
        }

        if not np.isnan(sens):
            sens_l.append(sens)
        if not np.isnan(spec):
            spec_l.append(spec)
        if not np.isnan(youden):
            yj_l.append(youden)

    return {
        "N": N,
        "threshold": threshold,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "hamming_accuracy": hamming_acc,
        "exact_match_accuracy": exact_match,
        "macro_sensitivity": float(np.mean(sens_l)) if sens_l else float("nan"),
        "macro_specificity": float(np.mean(spec_l)) if spec_l else float("nan"),
        "macro_youden_j": float(np.mean(yj_l)) if yj_l else float("nan"),
        "roc_auc_macro": float("nan"),
        "roc_auc_micro": float("nan"),
        "fabrication_error_rate": fabrication_error_rate,
        "fer_abnormal": fer_abnormal,
        "omission_rate": omission_rate,
        "total_fp_labels": total_fp_labels,
        "total_predicted_positive_labels": total_predicted_positive_labels,
        "total_fn_labels": total_fn_labels,
        "total_gt_positive_labels": total_gt_positive_labels,
        "num_abnormal_cases": num_abnormal_cases,
        "abnormal_cases_with_pred_positive": abnormal_cases_with_pred_positive,
        "per_label": per_label,
    }

def print_metrics_table(m: Dict[str, Any]) -> None:
    print("\n" + "=" * 96)
    print("  EVALUATION METRICS SUMMARY")
    print("=" * 96)
    print(f"  N samples                         : {m['N']}")
    print(f"  Threshold                         : {m['threshold']}")
    print(f"  Macro F1                          : {m['macro_f1']:.4f}")
    print(f"  Micro F1                          : {m['micro_f1']:.4f}")
    print(f"  Hamming Accuracy                  : {m['hamming_accuracy']:.4f}")
    print(f"  Exact Match Accuracy              : {m['exact_match_accuracy']:.4f}")
    print(f"  Macro Sensitivity                 : {m['macro_sensitivity']:.4f}")
    print(f"  Macro Specificity                 : {m['macro_specificity']:.4f}")
    print(f"  Macro Youden-J                    : {m['macro_youden_j']:.4f}")
    print(f"  ROC-AUC (macro)                   : {m['roc_auc_macro']}")
    print(f"  ROC-AUC (micro)                   : {m['roc_auc_micro']}")
    print(f"  Fabrication Error Rate (FER)      : {m['fabrication_error_rate']:.4f}")
    print(f"  FER for abnormal studies          : {m['fer_abnormal']:.4f}")
    print(f"  Omission Rate                     : {m['omission_rate']:.4f}")
    print()
    print(f"  Total FP labels                   : {m['total_fp_labels']}")
    print(f"  Total predicted positive labels   : {m['total_predicted_positive_labels']}")
    print(f"  Total FN labels                   : {m['total_fn_labels']}")
    print(f"  Total GT positive labels          : {m['total_gt_positive_labels']}")
    print(f"  # abnormal cases                  : {m['num_abnormal_cases']}")
    print(f"  # abnormal cases w/ pred positive : {m['abnormal_cases_with_pred_positive']}")

    H = f"\n  {'Label':<32} {'F1':>6} {'Sens':>6} {'Spec':>6} {'Youden':>7} {'AUC':>6}"
    print(H)
    print("  " + "-" * (len(H) - 3))
    for lab, lm in m["per_label"].items():
        print(
            f"  {lab:<32} "
            f"{lm['f1']:>6.3f} "
            f"{lm['sensitivity']:>6.3f} "
            f"{lm['specificity']:>6.3f} "
            f"{lm['youden_j']:>7.3f} "
            f"{lm.get('roc_auc', float('nan')):>6}"
        )
    print("=" * 96 + "\n")

In [16]:
# ------------------------------------------------------------
# Main evaluation
# ------------------------------------------------------------
pseudo_rows = load_jsonl(PSEUDO_REPORT_JSONL)
print(f"Loaded pseudo-report rows: {len(pseudo_rows)}")

all_y_true: List[np.ndarray] = []
all_y_pred: List[np.ndarray] = []

debug_rows: List[Dict[str, Any]] = []
missing_gt = 0
missing_pred_text = 0

for row in pseudo_rows:
    image_path = row.get("image_path", "")
    if not image_path:
        continue

    gt_json_path = image_path_to_json_path(image_path)
    if not gt_json_path.exists():
        missing_gt += 1
        print(f"⚠️ Missing GT json: {gt_json_path}")
        continue

    y_true_vec, gt_normal = extract_gt_labels_binary_from_json_path(gt_json_path)

    pseudo_report = row.get("pseudo_report", "")
    pred_text = extract_predicted_labels_text(pseudo_report)
    if not pred_text:
        missing_pred_text += 1

    y_pred_vec, pred_normal, pred_label_list = predicted_text_to_multihot(pred_text)

    all_y_true.append(y_true_vec)
    all_y_pred.append(y_pred_vec)

    debug_rows.append({
        "image_path": image_path,
        "gt_json_path": str(gt_json_path),
        "gt_normal": gt_normal,
        "pred_normal": pred_normal,
        "predicted_labels_text": pred_text,
        "predicted_labels_canonical": pred_label_list,
        "y_true_sum": int(y_true_vec.sum()),
        "y_pred_sum": int(y_pred_vec.sum()),
    })

if len(all_y_true) == 0:
    raise RuntimeError("No valid evaluation pairs were found.")

y_true = np.stack(all_y_true).astype(int)
y_pred = np.stack(all_y_pred).astype(int)

print(f"Valid evaluated samples: {len(y_true)}")
print(f"Missing GT json files  : {missing_gt}")
print(f"Missing pred text rows : {missing_pred_text}")

metrics = compute_comprehensive_metrics_hard_labels_only(
    y_true=y_true,
    y_pred=y_pred,
    threshold=0.5,
)
print_metrics_table(metrics)

print("Preview of parsed examples:")
for r in debug_rows[:5]:
    print("-" * 80)
    print("image_path              :", r["image_path"])
    print("gt_json_path            :", r["gt_json_path"])
    print("gt_normal               :", r["gt_normal"])
    print("pred_normal             :", r["pred_normal"])
    print("predicted_labels_text   :", r["predicted_labels_text"])
    print("predicted_labels_canon  :", r["predicted_labels_canonical"])

Loaded pseudo-report rows: 11539
Valid evaluated samples: 11539
Missing GT json files  : 0
Missing pred text rows : 0

  EVALUATION METRICS SUMMARY
  N samples                         : 11539
  Threshold                         : 0.5
  Macro F1                          : 0.1586
  Micro F1                          : 0.2750
  Hamming Accuracy                  : 0.8830
  Exact Match Accuracy              : 0.4495
  Macro Sensitivity                 : 0.1161
  Macro Specificity                 : 0.9814
  Macro Youden-J                    : 0.0975
  ROC-AUC (macro)                   : nan
  ROC-AUC (micro)                   : nan
  Fabrication Error Rate (FER)      : 0.4046
  FER for abnormal studies          : 0.2966
  Omission Rate                     : 0.8212

  Total FP labels                   : 2261
  Total predicted positive labels   : 5588
  Total FN labels                   : 15285
  Total GT positive labels          : 18612
  # abnormal cases                  : 5602
  # abnormal c

In [17]:
metrics

{'N': 11539,
 'threshold': 0.5,
 'macro_f1': 0.15863715822793634,
 'micro_f1': 0.2749586776859504,
 'hamming_accuracy': 0.8830321251674922,
 'exact_match_accuracy': 0.4495190224456192,
 'macro_sensitivity': 0.11606698536263639,
 'macro_specificity': 0.9814235508400265,
 'macro_youden_j': 0.09749053620266296,
 'roc_auc_macro': nan,
 'roc_auc_micro': nan,
 'fabrication_error_rate': 0.4046170365068003,
 'fer_abnormal': 0.29661733615221986,
 'omission_rate': 0.821244358478401,
 'total_fp_labels': 2261,
 'total_predicted_positive_labels': 5588,
 'total_fn_labels': 15285,
 'total_gt_positive_labels': 18612,
 'num_abnormal_cases': 5602,
 'abnormal_cases_with_pred_positive': 3273,
 'per_label': {'atelectasis': {'TP': 161,
   'FP': 170,
   'TN': 8517,
   'FN': 2691,
   'f1': 0.10116242538485705,
   'sensitivity': 0.056451612903206015,
   'specificity': 0.980430528375621,
   'youden_j': 0.03688214127882694,
   'roc_auc': nan},
  'cardiomegaly': {'TP': 1179,
   'FP': 1234,
   'TN': 7919,
   'FN':

In [13]:
import pandas as pd
from pathlib import Path
from typing import Dict, Any

def save_metrics_to_csv(
    metrics: Dict[str, Any],
    output_prefix: str,
) -> None:
    """
    Save evaluation metrics dict into CSV files.

    Generates:
      - {output_prefix}_summary.csv
      - {output_prefix}_per_label.csv
    """

    output_prefix = Path(output_prefix)

    # --------------------------------------------------------
    # 1) Summary metrics
    # --------------------------------------------------------
    summary_keys = [
        "N",
        "threshold",
        "macro_f1",
        "micro_f1",
        "hamming_accuracy",
        "exact_match_accuracy",
        "macro_sensitivity",
        "macro_specificity",
        "macro_youden_j",
        "roc_auc_macro",
        "roc_auc_micro",
        "fabrication_error_rate",
        "fer_abnormal",
        "omission_rate",
        "total_fp_labels",
        "total_predicted_positive_labels",
        "total_fn_labels",
        "total_gt_positive_labels",
        "num_abnormal_cases",
        "abnormal_cases_with_pred_positive",
    ]

    summary_data = {
        k: metrics.get(k, None) for k in summary_keys
    }

    df_summary = pd.DataFrame([summary_data])
    summary_path = output_prefix.with_name(output_prefix.name + "_summary.csv")
    df_summary.to_csv(summary_path, index=False)

    # --------------------------------------------------------
    # 2) Per-label metrics
    # --------------------------------------------------------
    per_label_data = []

    for label, values in metrics["per_label"].items():
        row = {"label": label}
        row.update(values)
        per_label_data.append(row)

    df_per_label = pd.DataFrame(per_label_data)
    per_label_path = output_prefix.with_name(output_prefix.name + "_per_label.csv")
    df_per_label.to_csv(per_label_path, index=False)

    print(f"✅ Saved summary metrics to: {summary_path}")
    print(f"✅ Saved per-label metrics to: {per_label_path}")

In [15]:
save_path = "/data/liangz2/openi/medgemma_predict"
save_metrics_to_csv(metrics, save_path + "/medgemma_val_metrics")

✅ Saved summary metrics to: /data/liangz2/openi/medgemma_predict/medgemma_val_metrics_summary.csv
✅ Saved per-label metrics to: /data/liangz2/openi/medgemma_predict/medgemma_val_metrics_per_label.csv


In [7]:
# ------------------------------------------------------------
# Optional: preview a few parsed examples
# ------------------------------------------------------------
print("Preview of parsed examples:")
for r in debug_rows[:5]:
    print("-" * 80)
    print("image_path              :", r["image_path"])
    print("gt_json_path            :", r["gt_json_path"])
    print("gt_normal               :", r["gt_normal"])
    print("pred_normal             :", r["pred_normal"])
    print("predicted_labels_text   :", r["predicted_labels_text"])
    print("predicted_labels_canon  :", r["predicted_labels_canonical"])

Preview of parsed examples:
--------------------------------------------------------------------------------
image_path              : /vf/users/liangz2/openi/mimic_train/50000766.jpg
gt_json_path            : /vf/users/liangz2/openi/mimic_train/50000766.json
gt_normal               : no
pred_normal             : yes
predicted_labels_text   : Normal chest X-ray
predicted_labels_canon  : []
--------------------------------------------------------------------------------
image_path              : /vf/users/liangz2/openi/mimic_train/50002196.jpg
gt_json_path            : /vf/users/liangz2/openi/mimic_train/50002196.json
gt_normal               : yes
pred_normal             : yes
predicted_labels_text   : Normal chest X-ray
predicted_labels_canon  : []
--------------------------------------------------------------------------------
image_path              : /vf/users/liangz2/openi/mimic_train/50002844.jpg
gt_json_path            : /vf/users/liangz2/openi/mimic_train/50002844.json
gt_normal